In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import torch 
import numpy as np 
from rcdiff.data.var import VAR

In [5]:
Phi_1 = np.array([[0.5,0.1,0], [0,.1,.2],[0,.3,.3]]).T
Phi_2 = np.array([[0.25,0.06,0.02], [0,.07,8e-2],[0,.12,.15]]).T
Phi_3 = np.array([[0.125,0.037,0.018], [0,.031,.038],[0,.057,.069]]).T
sigma_u = np.array([[1.0, 0.3, 0.1],
                    [0.3, 1.0, 0.2],
                    [0.1, 0.2, 1.0]])
A = [Phi_1, Phi_2, Phi_3]

In [6]:
var = VAR(A_coefs=A, sigma_u=sigma_u)
y = var.generate_trajectory(T = 1000, batch_size=1, burn_in=50)
y = y.squeeze(0)
print(y.shape) 
print(y)

torch.Size([1000, 3])
tensor([[ 0.0785, -1.3303, -2.2412],
        [ 1.4857, -0.8877, -0.7519],
        [ 0.8692, -0.0085, -2.5377],
        ...,
        [-1.2196,  0.4743, -1.4462],
        [-1.1444, -1.0275,  0.9968],
        [-0.2696, -0.7107, -0.7716]])


In [10]:
from rcdiff import ESN 
from rcdiff import RandomWeight, ParametersESN

R = 50 
T, N = y.shape
A = RandomWeight(dist="normal", sparsity=0.2,norm_type="spectral_radius")
C = RandomWeight(dist="uniform", sparsity=0.2,norm_type="l2")
zeta = RandomWeight(dist="uniform", sparsity=0.2,norm_type="euclidean")

esnParams = ParametersESN(A_RW = A, 
                 C_RW = C,
                 zeta_RW = zeta,
                 leak_rate = 0.2, 
                 rho = 0.1, 
                 gamma = 0.4,
                 omega = 0) 

esn = ESN(in_dim = N, 
          reservoir_size= R, 
          params=esnParams
          )

In [ ]:
esn